# Proyek Analisis Data: Air Quality Dataset - Beijing PM2.5
- **Nama:** Fahru Alfarizi
- **Email:** fahrualfarizi7@gmail.com
- **ID Dicoding:** fahp

## Menentukan Pertanyaan Bisnis
- Pertanyaan 1: Apakah terdapat perbedaan signifikan (>15%) rata-rata konsentrasi PM2.5 antara stasiun Urban dan Suburban selama musim dingin (Desember-Februari) tahun 2015-2016 di Beijing?
- Pertanyaan 2: Bagaimana korelasi antara faktor meteorological (kecepatan angin WSPM dan curah hujan RAIN) terhadap konsentrasi PM2.5 di seluruh stasiun pemantauan Beijing selama periode 2013-2017?

## Menyiapkan semua library yang dibutuhkan

In [ ]:
# Import library yang diperlukan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from datetime import datetime

# Set style visualisasi
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

# Menampilkan output matplotlib dalam notebook
%matplotlib inline

print('Library berhasil diimport!')

## Data Wrangling

### Gathering Data

In [ ]:
# Path ke folder dataset
data_path = 'data/'

# List semua file CSV
csv_files = [f for f in os.listdir(data_path) if f.endswith('.csv')]
print(f'Ditemukan {len(csv_files)} file CSV:\n')
for f in sorted(csv_files):
    print(f'  - {f}')

In [ ]:
# Load dan gabungkan semua data
all_data = []
for file in csv_files:
    df = pd.read_csv(os.path.join(data_path, file))
    all_data.append(df)
    
# Gabungkan menjadi satu DataFrame
df = pd.concat(all_data, ignore_index=True)
print(f'Total records: {len(df):,}')
print(f'\nShape: {df.shape}')
print(f'\nStasiun pemantauan: {df["station"].nunique()} stasiun')
df.head()

### Assessing Data

In [ ]:
# Info umum dataset
print('='*60)
print('INFO DATASET')
print('='*60)
df.info()

In [ ]:
# Statistik deskriptif
print('\n' + '='*60)
print('STATISTIK DESKRIPTIF')
print('='*60)
df.describe()

In [ ]:
# Cek missing values
print('='*60)
print('MISSING VALUES')
print('='*60)
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage (%)': missing_pct
}).sort_values('Missing Count', ascending=False)
missing_df[missing_df['Missing Count'] > 0]

In [ ]:
# Identifikasi missing values per stasiun
print('\nMISSING VALUES PM2.5 PER STASIUN:')
missing_by_station = df.groupby('station')['PM2.5'].apply(lambda x: x.isnull().sum())
print(missing_by_station.sort_values(ascending=False))

In [ ]:
# Cek duplicate data
print('='*60)
print('DUPLICATE DATA')
print('='*60)
print(f'Total duplicate rows: {df.duplicated().sum()}')

# Cek outlier menggunakan IQR
print('\nOUTLIER DETECTION (IQR Method):')
print('-'*40)
numeric_cols = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']

outlier_info = {}
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)][col]
    outlier_info[col] = {
        'count': len(outliers),
        'pct': round(len(outliers) / df[col].notna().sum() * 100, 2)
    }

outlier_df = pd.DataFrame(outlier_info).T
outlier_df.columns = ['Outlier Count', 'Percentage (%)']
outlier_df.sort_values('Outlier Count', ascending=False)

In [ ]:
# Cek validitas nilai
print('='*60)
print('VALIDITY CHECK')
print('='*60)

# PM2.5 tidak boleh negatif
neg_pm25 = df[df['PM2.5'] < 0]['PM2.5'].count()
print(f'PM2.5 negative values: {neg_pm25}')

# Hour harus 0-23
invalid_hour = df[(df['hour'] < 0) | (df['hour'] > 23)]['hour'].count()
print(f'Invalid hour values (not 0-23): {invalid_hour}')

# Month harus 1-12
invalid_month = df[(df['month'] < 1) | (df['month'] > 12)]['month'].count()
print(f'Invalid month values (not 1-12): {invalid_month}')

# Cek arah angin
print(f'\nWind direction (wd) unique values: {df["wd"].nunique()}')
print(f'Sample wd values: {df["wd"].dropna().unique()[:10]}')

### 1.3 Data Quality Issues Summary

Berdasarkan assessment di atas, ditemukan beberapa permasalahan:

| No | Permasalahan | Kolom | Solusi |
|---|--------------|------|--------|
| 1 | **Missing Values** | PM2.5, PM10, dll | Imputation dengan median per stasiun per bulan |
| 2 | **Outliers** | PM2.5, PM10, CO | Akan dipertahankan (natural phenomenon) |
| 3 | **Inconsistent value** | wd (direction) | Normalisasi menjadi 16 arah kompas |

### 1.4 Cleaning Data

### Cleaning Data

In [ ]:
# Buat salinan dataframe
df_clean = df.copy()

# 1. Handle missing values - Imputation dengan median per stasiun per bulan
print('='*60)
print('CLEANING: MISSING VALUES')
print('='*60)
print(f'Missing values BEFORE cleaning:')
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

# Fill missing values untuk setiap stasiun
for col in ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'RAIN', 'WSPM']:
    df_clean[col] = df_clean.groupby(['station', 'month'])[col].transform(
        lambda x: x.fillna(x.median())
    )
    # Jika masih ada missing (karena semua nilai dalam grup NaN), fill dengan median keseluruhan
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print(f'\nMissing values AFTER cleaning:')
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

In [ ]:
# 2. Buat datetime column
df_clean['datetime'] = pd.to_datetime(
    df_clean[['year', 'month', 'day', 'hour']].rename(columns={'day': 'Day'})
)
print('='*60)
print('DATETIME COLUMN CREATED')
print('='*60)
print(f'Date range: {df_clean["datetime"].min()} to {df_clean["datetime"].max()}')

# 3. Kategorikan stasiun sebagai Urban/Suburban
urban_stations = ['Aotizhongxin', 'Dongsi', 'Gucheng', 'Nongzhanguan', 'Tiantan', 'Wanshouxigong']
suburban_stations = ['Changping', 'Dingling', 'Huairou', 'Shunyi', 'Wanliu', 'Guanyuan']

df_clean['location_type'] = df_clean['station'].apply(
    lambda x: 'Urban' if x in urban_stations else 'Suburban'
)
print(f'\nUrban stations: {len(urban_stations)}')
print(f'Suburban stations: {len(suburban_stations)}')

In [ ]:
# Tampilkan sample data yang sudah dibersihkan
print('='*60)
print('SAMPLE CLEANED DATA')
print('='*60)
df_clean.head(10)

## Exploratory Data Analysis (EDA)

### Explore ...

In [ ]:
# Filter data untuk musim dingin 2015-2016
# Winter 2015-2016: Dec 2015, Jan 2016, Feb 2016
winter_data = df_clean[
    ((df_clean['year'] == 2015) & (df_clean['month'] == 12)) |
    ((df_clean['year'] == 2016) & (df_clean['month'].isin([1, 2])))
].copy()

print('='*60)
print('EDA PERTANYAAN BISNIS #1')
print('='*60)
print(f'Total records winter 2015-2016: {len(winter_data):,}')
print(f'Date range: {winter_data["datetime"].min()} to {winter_data["datetime"].max()}')

# Rata-rata PM2.5 per tipe lokasi
pm25_by_location = winter_data.groupby('location_type')['PM2.5'].agg(['mean', 'median', 'std', 'count'])
pm25_by_location = pm25_by_location.round(2)
pm25_by_location

In [ ]:
# Hitung persentase perbedaan
urban_mean = pm25_by_location.loc['Urban', 'mean']
suburban_mean = pm25_by_location.loc['Suburban', 'mean']
diff_pct = abs(urban_mean - suburban_mean) / suburban_mean * 100

print(f'\nRATA-RATA PM2.5 SELAMA MUSIM DINGIN 2015-2016:')
print(f'  Urban stations:    {urban_mean:.2f} µg/m³')
print(f'  Suburban stations:  {suburban_mean:.2f} µg/m³')
print(f'  Perbedaan:          {diff_pct:.2f}%')

if diff_pct > 15:
    print(f'\n✓ PERBEDAAN SIGNIFIKAN (>15%) TERDETEKSI!')
    print(f'  Urban memiliki konsentrasi PM2.5 {(urban_mean/suburban_mean - 1)*100:.1f}% lebih tinggi')
else:
    print(f'\n✗ Perbedaan TIDAK SIGNIFIKAN (<15%)')

In [ ]:
# Breakdown per stasiun
pm25_by_station = winter_data.groupby(['station', 'location_type'])['PM2.5'].mean().round(2).reset_index()
pm25_by_station = pm25_by_station.sort_values('PM2.5', ascending=False)
pm25_by_station

In [ ]:
# Korelasi untuk seluruh dataset
print('='*60)
print('EDA PERTANYAAN BISNIS #2')
print('='*60)

# Pilih kolom untuk korelasi
corr_cols = ['PM2.5', 'WSPM', 'RAIN', 'TEMP', 'PRES', 'DEWP']

# Hitung korelasi
correlation_matrix = df_clean[corr_cols].corr()

print('KORELASI DENGAN PM2.5:')
print('-'*40)
pm25_corr = correlation_matrix['PM2.5'].drop('PM2.5').sort_values(ascending=False)
for col, corr in pm25_corr.items():
    strength = 'Lemah'
    if abs(corr) > 0.5:
        strength = 'Kuat'
    elif abs(corr) > 0.3:
        strength = 'Sedang'
    direction = 'Positif' if corr > 0 else 'Negatif'
    print(f'  {col}: {corr:.3f} ({strength} {direction})')

In [ ]:
# Korelasi per tahun
yearly_corr = df_clean.groupby('year').apply(
    lambda x: x[['PM2.5', 'WSPM', 'RAIN']].corr()['PM2.5'][['WSPM', 'RAIN']]
).T

print('\nKORELASI WSPM & RAIN vs PM2.5 PER TAHUN:')
print('-'*40)
yearly_corr.round(3)

In [ ]:
# Analisis berdasarkan kecepatan angin
print('\nRATA-RATA PM2.5 BERDASARKAN KATEGORI KECEPATAN ANGIN:')
print('-'*40)

# Kategori kecepatan angin
def categorize_wspm(wspm):
    if wspm < 2:
        return 'Calm (<2 m/s)'
    elif wspm < 4:
        return 'Light (2-4 m/s)'
    elif wspm < 6:
        return 'Moderate (4-6 m/s)'
    else:
        return 'Strong (>6 m/s)'

df_clean['wspm_category'] = df_clean['WSPM'].apply(categorize_wspm)
wspm_analysis = df_clean.groupby('wspm_category')['PM2.5'].agg(['mean', 'median', 'std', 'count']).round(2)
wspm_analysis

## Visualization & Explanatory Analysis

### Pertanyaan 1: Apakah terdapat perbedaan signifikan (>15%) rata-rata konsentrasi PM2.5 antara stasiun Urban dan Suburban selama musim dingin (Desember-Februari) tahun 2015-2016 di Beijing?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 1. Bar chart perbandingan
ax1 = axes[0]
colors = ['#e74c3c', '#3498db']  # Urban=red, Suburban=blue
bars = ax1.bar(pm25_by_location.index, pm25_by_location['mean'],
        color=colors, edgecolor='black', linewidth=1.5)

# Tambahkan nilai di atas bar
for bar, val in zip(bars, pm25_by_location['mean']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
             f'{val:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=12)

ax1.set_ylabel('Rata-rata PM2.5 (µg/m³)', fontsize=11)
ax1.set_title('Rata-rata PM2.5: Urban vs Suburban\nMusim Dingin 2015-2016', fontsize=13, fontweight='bold')
ax1.set_ylim(0, max(pm25_by_location['mean']) * 1.15)

# 2. Box plot distribusi
ax2 = axes[1]
bp = ax2.boxplot([
    winter_data[winter_data['location_type'] == 'Urban']['PM2.5'],
    winter_data[winter_data['location_type'] == 'Suburban']['PM2.5']
], labels=['Urban', 'Suburban'], patch_artist=True,
    boxprops=dict(facecolor='lightblue', color='navy'),
    medianprops=dict(color='red', linewidth=2))

bp['boxes'][0].set_facecolor('#e74c3c')
bp['boxes'][1].set_facecolor('#3498db')

ax2.set_ylabel('PM2.5 (µg/m³)', fontsize=11)
ax2.set_title('Distribusi PM2.5: Urban vs Suburban\nMusim Dingin 2015-2016', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('viz_pm25_urban_suburban.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✓ Visualisasi disimpan: viz_pm25_urban_suburban.png')

In [ ]:
# Breakdown per stasiun (top 6 dan bottom 6)
fig, ax = plt.subplots(figsize=(12, 6))

pm25_station_sorted = pm25_by_station.sort_values('PM2.5', ascending=True)
colors = ['#3498db' if loc == 'Suburban' else '#e74c3c' 
          for loc in pm25_station_sorted['location_type']]

bars = ax.barh(pm25_station_sorted['station'], pm25_station_sorted['PM2.5'],
       color=colors, edgecolor='black', linewidth=0.8)

ax.set_xlabel('Rata-rata PM2.5 (µg/m³)', fontsize=11)
ax.set_title('Rata-rata PM2.5 per Stasiun Pemantauan\nMusim Dingin 2015-2016', fontsize=13, fontweight='bold')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#e74c3c', label='Urban'),
                   Patch(facecolor='#3498db', label='Suburban')]
ax.legend(handles=legend_elements, loc='lower right')

# Tambahkan nilai
for i, (bar, val) in enumerate(zip(bars, pm25_station_sorted['PM2.5'])):
    ax.text(val + 2, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=9)

plt.tight_layout()
plt.savefig('viz_pm25_by_station.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✓ Visualisasi disimpan: viz_pm25_by_station.png')

### Pertanyaan 2: Bagaimana korelasi antara faktor meteorological (kecepatan angin WSPM dan curah hujan RAIN) terhadap konsentrasi PM2.5 di seluruh stasiun pemantauan Beijing selama periode 2013-2017?

In [ ]:
# 1. Correlation Heatmap
fig, ax = plt.subplots(figsize=(8, 6))

mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
sns.heatmap(correlation_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Correlation Coefficient'})

ax.set_title('Correlation Matrix: PM2.5 & Meteorological Factors\nBeijing 2013-2017', 
             fontsize=13, fontweight='bold', pad=15)

plt.tight_layout()
plt.savefig('viz_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✓ Visualisasi disimpan: viz_correlation_heatmap.png')

In [ ]:
# 2. Scatter plot WSPM vs PM2.5
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter plot WSPM vs PM2.5
ax1 = axes[0]
sample = df_clean.sample(n=min(5000, len(df_clean)), random_state=42)
scatter = ax1.scatter(sample['WSPM'], sample['PM2.5'],
                      c=sample['year'], cmap='viridis', alpha=0.4, s=10)
ax1.set_xlabel('Kecepatan Angin (m/s)', fontsize=11)
ax1.set_ylabel('PM2.5 (µg/m³)', fontsize=11)
ax1.set_title('Hubungan Kecepatan Angin vs PM2.5\nBeijing 2013-2017', fontsize=13, fontweight='bold')
plt.colorbar(scatter, ax=ax1, label='Tahun')

# Scatter plot RAIN vs PM2.5
ax2 = axes[1]
scatter2 = ax2.scatter(sample['RAIN'], sample['PM2.5'],
                       c=sample['year'], cmap='viridis', alpha=0.4, s=10)
ax2.set_xlabel('Curah Hujan (mm)', fontsize=11)
ax2.set_ylabel('PM2.5 (µg/m³)', fontsize=11)
ax2.set_title('Hubungan Curah Hujan vs PM2.5\nBeijing 2013-2017', fontsize=13, fontweight='bold')
plt.colorbar(scatter2, ax=ax2, label='Tahun')

plt.tight_layout()
plt.savefig('viz_meteo_scatter.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✓ Visualisasi disimpan: viz_meteo_scatter.png')

In [ ]:
# 3. Bar chart rata-rata PM2.5 berdasarkan kategori kecepatan angin
fig, ax = plt.subplots(figsize=(10, 6))

wspm_order = ['Calm (<2 m/s)', 'Light (2-4 m/s)', 'Moderate (4-6 m/s)', 'Strong (>6 m/s)']
wspm_colors = ['#c0392b', '#e74c3c', '#f39c12', '#27ae60']

bars = ax.bar(wspm_analysis.loc[wspm_order].index, 
             wspm_analysis.loc[wspm_order, 'mean'],
             color=wspm_colors, edgecolor='black', linewidth=1.5)

for bar, val in zip(bars, wspm_analysis.loc[wspm_order, 'mean']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
            f'{val:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=11)

ax.set_xlabel('Kategori Kecepatan Angin', fontsize=11)
ax.set_ylabel('Rata-rata PM2.5 (µg/m³)', fontsize=11)
ax.set_title('Rata-rata PM2.5 Berdasarkan Kategori Kecepatan Angin\nBeijing 2013-2017', 
            fontsize=13, fontweight='bold')
ax.set_ylim(0, wspm_analysis['mean'].max() * 1.15)

plt.tight_layout()
plt.savefig('viz_pm25_by_wspm.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n✓ Visualisasi disimpan: viz_pm25_by_wspm.png')

## Conclusion

### Pertanyaan 1: Apakah terdapat perbedaan signifikan (>15%) rata-rata konsentrasi PM2.5 antara stasiun Urban dan Suburban selama musim dingin (Desember-Februari) tahun 2015-2016 di Beijing?

**Jawaban:**
YA, terdapat perbedaan signifikan.

| Tipe Lokasi | Rata-rata PM2.5 | Median |
|-------------|-----------------|--------|
| **Urban**   | 93.54 µg/m³     | 43.00 µg/m³ |
| **Suburban**| 80.72 µg/m³     | 34.00 µg/m³ |

**Temuan Utama:**
- Konsentrasi PM2.5 di stasiun Urban adalah **93.54 µg/m³**, sedangkan di Suburban adalah **80.72 µg/m³**.
- PM2.5 di stasiun Urban **15.87% lebih tinggi** dibanding Suburban, yang melebihi ambang batas signifikansi 15%.
- Pola ini konsisten dengan karakteristik polusi perkotaan Beijing, di mana area urban memiliki aktivitas industri, lalu lintas, dan kepadatan penduduk yang lebih tinggi.

### Pertanyaan 2: Bagaimana korelasi antara faktor meteorological (kecepatan angin WSPM dan curah hujan RAIN) terhadap konsentrasi PM2.5 di seluruh stasiun pemantauan Beijing selama periode 2013-2017?

**Jawaban:**
Keduanya memiliki korelasi negatif, dengan rincian sebagai berikut:

| Faktor Meteorologi | Koefisien Korelasi (r) | Interpretasi |
|--------------------|------------------------|--------------|
| **Kecepatan Angin (WSPM)** | -0.269 | Korelasi negatif lemah-sedang |
| **Curah Hujan (RAIN)** | -0.014 | Korelasi negatif sangat lemah (tidak signifikan) |

**Temuan Utama:**
- **Kecepatan Angin (WSPM)** memiliki korelasi negatif sebesar **-0.269** dengan PM2.5. Hal ini menunjukkan bahwa angin kencang berkontribusi signifikan dalam mendispersi partikel PM2.5. Rata-rata PM2.5 pada kondisi tenang (<2 m/s) mencapai **92.42 µg/m³**, sedangkan pada kondisi angin kencang (>6 m/s) rata-rata PM2.5 turun menjadi **26.73 µg/m³** (penurunan sebesar **71.1%**).
- **Curah Hujan (RAIN)** memiliki korelasi negatif yang sangat lemah sebesar **-0.014** dengan PM2.5, menunjukkan bahwa keberadaan hujan secara statistik tidak berkorelasi kuat dengan penurunan PM2.5 di seluruh stasiun secara agregat harian/jam.

In [ ]:
# Simpan data yang sudah dibersihkan untuk dashboard
df_clean.to_csv('cleaned_air_quality_data.csv', index=False)
print('✓ Data berhasil disimpan: cleaned_air_quality_data.csv')

# Simpan summary statistics
summary_stats = df_clean.groupby('station').agg({
    'PM2.5': ['mean', 'median', 'std'],
    'PM10': ['mean', 'median'],
    'TEMP': 'mean',
    'WSPM': 'mean'
}).round(2)
summary_stats.to_csv('station_summary_stats.csv')
print('✓ Summary statistics disimpan: station_summary_stats.csv')

In [ ]:
# Verifikasi file yang disimpan
print('\nFILES YANG DISIMPAN:')
print('-'*40)
for f in ['viz_pm25_urban_suburban.png', 'viz_pm25_by_station.png', 
          'viz_correlation_heatmap.png', 'viz_meteo_scatter.png',
          'viz_pm25_by_wspm.png', 'cleaned_air_quality_data.csv',
          'station_summary_stats.csv']:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024
        print(f'  ✓ {f} ({size:.1f} KB)')
    else:
        print(f'  ✗ {f} (not found)')